In [ ]:
# !pip install uv
# !uv pip install  -r requirements.txt

In [1]:
import time
# import snowflake
# from snowflake.snowpark.context import get_active_session
# session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os
import time

In [2]:
# Setup
tqdm.pandas()

catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

bands_of_interest = {'red', 'blue', 'drad', 'emis', 'emsd', 'lwir', 'trad', 'urad', 'atran', 'cdist', 'green', 'nir08', 'swir16', 'swir22', 'atmos_opacity'}

def compute_Landsat_values(row):
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')


    # Buffer size for ~100m 
    bbox_size = 0.00089831  
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    # Wider search range, we'll filter to nearest date later
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    
    items = search.item_collection()

    if not items:
        # return pd.Series({
        #     "nir": np.nan
        #     , "green": np.nan
        #     , "swir16": np.nan
        #     , "swir22": np.nan
        #     , "red": np.nan
        #     , "blue": np.nan
        # })
        print("No items found")
        return pd.Series([np.nan] * len(bands_of_interest), index=bands_of_interest)


    # Convert sample date to UTC
    sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

    # Pick the item closest to the sample date
    items = sorted(
        items,
        key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
    )
    selected_item = pc.sign(items[0])

    # max_key_length = len(max(selected_item.assets, key=len))
    # for key, asset in selected_item.assets.items():
    #     print(f"{key.rjust(max_key_length)}: {asset.title}")

    # Load required bands
    # bands_of_interest = ["green", "nir08", "swir16", "swir22", "red", "blue"]
    bands_to_extract = [band for band in bands_of_interest if band in selected_item.assets.keys()]
    data = stac_load([selected_item], bands=bands_to_extract, bbox=bbox).isel(time=0)


    medians = []
    for band_name in bands_of_interest:
        if band_name in selected_item.assets.keys():
            band = data[band_name].astype("float")
            # Compute medians
            median_band = float(band.median(skipna=True).values)
            # Replace 0 with NaN
            median_band = median_band if median_band != 0 else np.nan
            medians.append(median_band)
        else:
            medians.append(np.nan)
    # print(pd.Series(medians, index=bands_of_interest))
    return pd.Series(medians, index=bands_of_interest)

In [3]:
Water_Quality_df=pd.read_csv('../water_quality_training_dataset.csv')
display(Water_Quality_df.head())

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [4]:
start_index = 6000
end_index = 7000
start_end_index_diff = end_index - start_index
to_process_df = Water_Quality_df[start_index:end_index]
print(to_process_df.shape)

(1000, 6)


In [5]:
len_full_df = 0
count=0
file_path = f"all_bands_extraction/landsat_features_training_all_bands_{start_index}_{end_index}.csv"
full_df = None
while len_full_df < start_end_index_diff and count < 20:

    file_path = f"all_bands_extraction/landsat_features_training_all_bands_{start_index}_{end_index}.csv"
    full_df = None
    if os.path.isfile(file_path):
        full_df = pd.read_csv(file_path)

    len_to_process_df = to_process_df.shape[0]

    for index, row in to_process_df.iterrows():
        try:
            lon = row["Longitude"]
            lat = row["Latitude"]
            sample_date = row["Sample Date"]
            print(f"{index - start_index + 1}/{len_to_process_df} Processing\n\t- Lat: {lat}\n\t- Lon: {lon}\n\t- Sample Date: {sample_date}\n")

            if full_df is not None and index - start_index + 1 <= full_df.shape[0]:
                len_full_df = full_df.shape[0]
                print(f"\tSkipping... {index - start_index + 1} <= {full_df.shape[0]}")
                continue

            new_row = compute_Landsat_values(row)

            row_df = row.to_frame().T
            row_df = row_df[["Latitude", "Longitude", "Sample Date"]]
            new_row_df = new_row.to_frame().T

            row_df = row_df.reset_index(drop=True)
            new_row_df = new_row_df.reset_index(drop=True)
            new_row_df = pd.concat([row_df, new_row_df], axis=1)

            # print(new_row)
            if full_df is None:
                full_df = new_row_df
                print(full_df)
            else:
                full_df = full_df.reset_index(drop=True)
                full_df = pd.concat([full_df, new_row_df], axis=0, ignore_index=True)

            len_full_df = full_df.shape[0]

            print(f"len_full_df = {len_full_df}, count = {count}")
        except Exception as e:
            full_df.to_csv(file_path, index = False, mode='w')
            print(f"Failed !!!\n{e}")
            count+=1
            break

full_df.to_csv(file_path, index = False, mode='w')
read_df = pd.read_csv(file_path)
print(f"Verifying Data Frame # Rows: {read_df.shape[0]}")

1/1000 Processing
	- Lat: -28.801278
	- Lon: 31.955389
	- Sample Date: 27-05-2014

	Skipping... 1 <= 1000
2/1000 Processing
	- Lat: -26.970278
	- Lon: 27.211111
	- Sample Date: 27-05-2014

	Skipping... 2 <= 1000
3/1000 Processing
	- Lat: -26.98472222
	- Lon: 26.63227778
	- Sample Date: 27-05-2014

	Skipping... 3 <= 1000
4/1000 Processing
	- Lat: -33.501667
	- Lon: 21.624167
	- Sample Date: 27-05-2014

	Skipping... 4 <= 1000
5/1000 Processing
	- Lat: -25.45963889
	- Lon: 28.26430556
	- Sample Date: 27-05-2014

	Skipping... 5 <= 1000
6/1000 Processing
	- Lat: -25.81048333
	- Lon: 27.90955222
	- Sample Date: 28-05-2014

	Skipping... 6 <= 1000
7/1000 Processing
	- Lat: -22.935
	- Lon: 28.004167
	- Sample Date: 28-05-2014

	Skipping... 7 <= 1000
8/1000 Processing
	- Lat: -24.282222
	- Lon: 28.090278
	- Sample Date: 28-05-2014

	Skipping... 8 <= 1000
9/1000 Processing
	- Lat: -23.763056
	- Lon: 27.908611
	- Sample Date: 28-05-2014

	Skipping... 9 <= 1000
10/1000 Processing
	- Lat: -28.747778

In [ ]:
# session.sql(f"""
#     PUT file://{file_path}
#     'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge-v2"/versions/live/'
#     AUTO_COMPRESS=FALSE
#     OVERWRITE=TRUE
# """).collect()
#
# print("File saved! Refresh the browser to see the files in the sidebar")